# Project Phase 2

## Short Introduction to our Dataset

### Original Research Question

*"How heavily did a state’s COVID-19 measurement policy have an impact on mental health
compared to its counterparts?"*

### Dataset Descriptions

- 1. CDC Depression Data *(data.cdc.gov)*: Surveys about the mental health of the U.S. population from 2020 until 2024. This was done by asking the experienced depression and anxiety of U.S. citizens and categorized by sex, ethnicity, state, and age.
- 2. Covid Policy Dataset *(github.com/OxCGRT)*: An Oxford study that broke down numerical representation of the U.S. country-wide and state specific Covid-related policies. Recorded from 2020 until 2022 and in multiple policy categories.

### Joins Explained

```python
merged = converted_depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['RegionName', 'Start', 'End'],
    how='left'
)
```

We wanted to add the columns from the Covid Policy Dataset to our CDC Depression Data. We did this for all unique combinations of specific U.S. states and the time. Before we were able to do this, we needed to adjust the Covid Policy Dataset in multiple ways:

1) Fill 'United States' in the 'RegionName' column for all country-level data, as the Depression Dataset expects it in that format *(shown in step 1.3)*.
2) Group entries from daily entries to the time periods used in the Covid Policy Dataset *(shown in step 1.4)*.

After the merge we had all ~16,000 entries of the Depression dataset. ~12,000 of them were extended with data from the Covid Policy Dataset, which covers all entries from 2020 until 2022. The entries for 2023 and 2024 were not extended, as the OxCGRT dataset did not provide values for these years. We decided to do a left join instead of an inner join, as it can give good insight about the depression and anxiety rates development after the pandemic, even when we don't have specific insight into the regulations at this time.

### Final Dataset Shape (rows x columns)

- Columns: 42
- Rows: 16,794


## Part 1 - Dataset Preparation and Joins

We used a left join on the depression dataset on the columns 'State', 'Time Period Start Date', 'Time Period End Date' with matching columns on the Covid policy dataset. We needed to create matching start and end date columns on the Covid policy dataset first, as its records were daily, while the depression dataset recorded their findings in specific time periods. We did a left join, because the depression dataset contains the main information components that we want to use, is more granular in categories (such as records by sex or ethnicity) and we just want to add some information about the government policies at that time from the other dataset.

The depression dataset originally had 14 columns and grew by 28 columns. This happened as we added most of the columns of the Covid policy dataset. We filtered out some columns that we do not require or that already exist in the depression dataset, else we would have added even more columns.

### 1.1 Download the datasets

In [25]:
# Download the required datasets
import os
import urllib.request

os.makedirs("dataset", exist_ok=True)

file_path_1 = "dataset/cdc_depression_data.csv"
file_path_2 = "dataset/OxCGRT_simplified_v1.csv"
file_path_3 = "dataset/merged_dataset.csv"
url_path_1 = "https://data.cdc.gov/api/views/8pt5-q6wp/rows.csv?accessType=DOWNLOAD"
url_path_2 = "https://github.com/OxCGRT/covid-policy-dataset.git"

if not os.path.exists(file_path_1):
    print("Downloading CDC depression dataset from data.gov ...")
    urllib.request.urlretrieve(
        url_path_1,
        file_path_1
    )
else:
    print("Dataset already exists. Skipping download.")

if not os.path.exists(file_path_2):
    print("Downloading OxCGRT dataset from GitHub ...")
    # Clone the repository and move the file
    os.system("git clone " + url_path_2)
    os.system("mv covid-policy-dataset/data/OxCGRT_simplified_v1.csv dataset/")
    os.system("rm -rf covid-policy-dataset")
else:
    print("Dataset already exists. Skipping download.")


Dataset already exists. Skipping download.
Dataset already exists. Skipping download.


### 1.2 Load the Datasets using Pandas

In [26]:
# Load the datasets
import pandas as pd
import numpy as np

depression_data = pd.read_csv(file_path_1, low_memory=False)
oxcgrt_data = pd.read_csv(file_path_2, low_memory=False)

### 1.3 - Clean up and prepare our OxCGRT Dataset

In [27]:
# Filter the simplified OxCGRT dataset for the United States
oxcgrt_us = oxcgrt_data[oxcgrt_data['CountryName'] == 'United States']

# Remove some unnecessary columns that we won't be using for our analysis
columns_to_drop = ['CountryName', 'CountryCode', 'RegionCode', 'Jurisdiction']
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Our dataset contains two versions of the measurements: *_combined and *_combined_numeric.
# For this assignment we will only keep the *_combined_numeric columns, as they are easier to work with for analysis and visualization.
# We might want to include the *_combined columns in a future assignment when we do more detailed analysis
columns_to_drop = [col for col in oxcgrt_us.columns if col.endswith('_combined') and not col.endswith('_combined_numeric')]
oxcgrt_us = oxcgrt_us.drop(columns=columns_to_drop)

# Fill missing values in the 'RegionName' column with 'United States'
# Required for merging with the depression dataset, which does expect 'United States' for country-level data
oxcgrt_us.loc[oxcgrt_us['RegionName'].isna() | (oxcgrt_us['RegionName'] == ''), 'RegionName'] = 'United States'

# Print out all unique RegionName values
print("Unique RegionName values in the OxCGRT dataset:")
print(oxcgrt_us['RegionName'].unique())

# Rename all rows of 'RegionName' from 'Washington DC' to 'District of Columbia'
# We discovered at inspection, that a mismatching name between the two datasets was causing issues with merging
oxcgrt_us.loc[oxcgrt_us['RegionName'] == 'Washington DC', 'RegionName'] = 'District of Columbia'


oxcgrt_us.to_csv('dataset/oxcgrt_us.csv', index=False)


Unique RegionName values in the OxCGRT dataset:
<StringArray>
[ 'United States',         'Alaska',        'Alabama',       'Arkansas',
        'Arizona',     'California',       'Colorado',    'Connecticut',
  'Washington DC',       'Delaware',        'Florida',        'Georgia',
         'Hawaii',           'Iowa',          'Idaho',       'Illinois',
        'Indiana',         'Kansas',       'Kentucky',      'Louisiana',
  'Massachusetts',       'Maryland',          'Maine',       'Michigan',
      'Minnesota',       'Missouri',    'Mississippi',        'Montana',
 'North Carolina',   'North Dakota',       'Nebraska',  'New Hampshire',
     'New Jersey',     'New Mexico',         'Nevada',       'New York',
           'Ohio',       'Oklahoma',         'Oregon',   'Pennsylvania',
   'Rhode Island', 'South Carolina',   'South Dakota',      'Tennessee',
          'Texas',           'Utah',       'Virginia',        'Vermont',
     'Washington',      'Wisconsin',  'West Virginia',        

### 1.4 Aggregate the OxCGRT Dataset to match the Depression Datasets' Time Periods

In [28]:
# Get unique time periods and state combinations from the depression dataset
unique_time_state_combs = depression_data[['Time Period Start Date', 'Time Period End Date', 'State']].drop_duplicates()

# Convert 'Time Period Start Date' and 'Time Period End Date' to int YYYYMMDD format
unique_time_state_combs['Time Period Start Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
unique_time_state_combs['Time Period End Date'] = pd.to_datetime(
    unique_time_state_combs['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

# Sort the rows of the OxCGRT dataset into groups, based on the unique time-state combinations from the depression dataset
# Add 'Start' and 'End' columns to the OxCGRT dataset and fill them based on the unique time-state combinations
# These will be matched to the depression dataset later when we do the merge, so we can easily filter OxCGRT data for the relevant time periods and states
oxcgrt_us['Start'] = np.nan
oxcgrt_us['End'] = np.nan

for start, end, state in unique_time_state_combs.values:
    mask = (
        (oxcgrt_us['RegionName'] == state) &
        (oxcgrt_us['Date'] >= start) &
        (oxcgrt_us['Date'] <= end)
    )
    oxcgrt_us.loc[mask, 'Start'] = start
    oxcgrt_us.loc[mask, 'End'] = end

# Now we can drop the 'Date' column, as we have the 'Start' and 'End' columns to indicate the relevant time periods for each row
oxcgrt_us = oxcgrt_us.drop(columns=['Date'])

# Convert the 'PopulationVaccinated' column to numeric
# (It was imported as an object, need to convert it to numeric for aggregation)
oxcgrt_us['PopulationVaccinated'] = pd.to_numeric(oxcgrt_us['PopulationVaccinated'])

# Aggregate the OxCGRT data by group (unique time-state combinations) using median for numeric columns
oxcgrt_aggregated = (
    oxcgrt_us
    .dropna(subset=['Start', 'End'])
    .groupby(['RegionName', 'Start', 'End'], as_index=False)
    .median(numeric_only=True)
)

# Add MajorityVaccinated back SAFELY (no .values)
# We did not list this merge in our explanation in the notebook, as it is not a merge between our different datasets
# We just add the 'MajorityVaccinated' column back to the aggregated dataset, as it is a non-numeric column that we cannot aggregate using median
majority = (
    oxcgrt_us
    .dropna(subset=['Start', 'End'])
    .groupby(['RegionName', 'Start', 'End'], as_index=False)['MajorityVaccinated']
    .first()
)

oxcgrt_aggregated = oxcgrt_aggregated.merge(
    majority, on=['RegionName', 'Start', 'End'], how='left'
)

oxcgrt_aggregated.to_csv('dataset/oxcgrt_aggregated.csv', index=False)

### 1.5 Merge the two Datasets

In [29]:
# Now we can merge the depression dataset with the aggregated OxCGRT dataset based on the unique time-state combinations
# Beforehand we need to convert the 'Time Period Start Date' and 'Time Period End Date' columns in the depression dataset (such as in the cell before)
converted_depression_data = depression_data.copy()
converted_depression_data['Time Period Start Date'] = pd.to_datetime(
    converted_depression_data['Time Period Start Date'], format='%m/%d/%Y'
).dt.strftime('%Y%m%d').astype('int64')
converted_depression_data['Time Period End Date'] = pd.to_datetime(
    converted_depression_data['Time Period End Date'], format='%m/%d/%Y', errors='coerce'
).dt.strftime('%Y%m%d').astype('int64')

rows_before_left = len(converted_depression_data)
rows_before_right = len(oxcgrt_aggregated)

# Merge the datasets based on the state and time period columns
merged = converted_depression_data.merge(
    oxcgrt_aggregated,
    left_on=['State', 'Time Period Start Date', 'Time Period End Date'],
    right_on=['RegionName', 'Start', 'End'],
    how='left'
)

rows_after = len(merged)

print("Rows before merge (depression):", rows_before_left)
print("Rows before merge (policy aggregated):", rows_before_right)
print("Rows after merge:", rows_after)

# Clean up merged dataset
merged = merged.drop(columns=['RegionName', 'Start', 'End']) # Redundant after merge
# merged = merged.drop(columns=['Time Period Start Date', 'Time Period End Date']) # Optional, as the Time Period Label already indicates the time period

# Save the merged dataset to a new CSV file
merged.to_csv(file_path_3, index=False)
print("Final merged dataset shape (rows, cols):", merged.shape)


Rows before merge (depression): 16794
Rows before merge (policy aggregated): 2713
Rows after merge: 16794
Final merged dataset shape (rows, cols): (16794, 42)


### 1.6 Describe the original Datasets and the merged Dataset for comparison

In [30]:
print("Depression dataset:")
print(depression_data.shape)
print(depression_data.info())
print(depression_data.describe())
print("##########################################################################################")

print("\nOxCGRT dataset (filtered for United States):")
print(oxcgrt_us.shape)
print(oxcgrt_us.info())
print(oxcgrt_us.describe())
print("##########################################################################################")

print("\nMerged dataset:")
print(merged.shape)
print(merged.info())
print(merged.describe())


Depression dataset:
(16794, 14)
<class 'pandas.DataFrame'>
RangeIndex: 16794 entries, 0 to 16793
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Indicator               16794 non-null  str    
 1   Group                   16794 non-null  str    
 2   State                   16794 non-null  str    
 3   Subgroup                16794 non-null  str    
 4   Phase                   16794 non-null  str    
 5   Time Period             16794 non-null  int64  
 6   Time Period Label       16794 non-null  str    
 7   Time Period Start Date  16794 non-null  str    
 8   Time Period End Date    16794 non-null  str    
 9   Value                   16087 non-null  float64
 10  Low CI                  16087 non-null  float64
 11  High CI                 16087 non-null  float64
 12  Confidence Interval     16087 non-null  str    
 13  Quartile Range          11017 non-null  str    
dtypes: float64(3), in

## Part 2: Exploratory Questions + Interpretation

You must answer at least four specific EDA questions using visualization.

Each question must:

- Be clearly stated
- Use Pandas operations
- Include one iplot() visualization

For each visualization, answer the following in full sentences (3-5):

1. What question are you asking?
2. What method did you use?
3. What does the visualization show?
4. What insight can you draw from it?

---

**Optional - for future reports**

- Identifying rows lost during joins
- Using Boolean filtering or query()
- Identifying missing data patterns


In [31]:
# Prepare data visualization
import plotly
import cufflinks as cf

import plotly.express as px

# Load the merged dataset for visualization
# Requires execution of all previous cells to create the merged dataset first
merged_data = pd.read_csv(file_path_3, low_memory=False)


### 2.1 Category Counts

*"Which restriction level for C1, C2, C6 appears most frequently across the dataset?"*

Short comment on the categories and levels:

- **C1 School Closing** - Measures the severity of school closure policies, from no measures (0) to complete closure of all education levels (3).  
- **C2 Workplace Closing** - Captures how strictly workplaces were restricted, ranging from no measures (0) to closure of all non-essential workplaces (3).
- **C6 Stay-at-Home Requirements** - Indicates the strictness of stay-at-home orders, from no restrictions (0) to strict confinement with minimal exceptions (3).

We took a closer look at these three categories because we assumed they might have had the strongest impact on the population’s mental health. Using Pandas, we counted the occurrences of each policy level and visualized the distributions. The visualization shows that school closings were generally the strictest policy, with most values ranging between 1.5 and 2.5. Workplace closings were somewhat less strict but still frequently appeared in the 1, 1.5, and 2 categories. Stay-at-home requirements were the least strict overall, with most observations in the 0, 0.5, and 1 categories. This could suggest that younger populations—such as children, teenagers, and young adults—were more affected by COVID-related policies than older generations.



In [ ]:
# 1. Category Counts (Bar Chart)
levels = [0, 0.5, 1, 1.5, 2, 2.5, 3]

merged_data['C1M_combined_numeric'] \
    .value_counts() \
    .reindex(levels, fill_value=0) \
    .iplot(kind='bar',
           title='C1M School Closing Levels',
           xTitle='Restriction Level',
           yTitle='Frequency',) # type: ignore

merged_data['C2M_combined_numeric'] \
    .value_counts() \
    .reindex(levels, fill_value=0) \
    .iplot(kind='bar',
           title='C2M Workplace Closing Levels',
           xTitle='Restriction Level',
           yTitle='Frequency',) # type: ignore

merged_data['C6M_combined_numeric'] \
    .value_counts() \
    .reindex(levels, fill_value=0) \
    .iplot(kind='bar',
           title='C6M Stay at Home Requirements Levels',
           xTitle='Restriction Level',
           yTitle='Frequency',) # type: ignore

### 2.2 Grouped Aggregation

*"Which U.S. state had the highest average depression value?"*

We grouped by the 'state' column, took the mean for 'Value' and sorted them by the calculated average. We can see that Louisiana, Mississippi, and Oklahoma had the highest depression rates, while Wisconsin, Minnesota, and South Dakota the lowest. There is no sharp jump from one state to another; they have slow, gradually increasing depression rates. But this accumulates to an approximately 11% difference between Louisiana and South Dakota. This tells us that there is a major difference in experienced depression across U.S. states. However, this analysis does not allow us to determine whether COVID-related policies contributed to these differences or whether the higher rates already existed before the pandemic.

In [33]:
# Group by state and calculate the average depression value for each state
state_depression_avg = merged_data.groupby('State')['Value'].mean().sort_values(ascending=False)
state_depression_avg.iplot(kind='bar',
                             title='Average Depression Value by State',
                             xTitle='State',
                             yTitle='Average Depression Value',)

### 2.3 Distribution Histogram

*"How are depression values distributed among 18–29-year-olds across all time periods, and how much variability is present?"*

The histogram itself contains the depression values we filtered for (that is, only the 19-29 year olds for each of the time periods we filtered on before)․ It essentially allows us to visualize how often each value has occurred, and how concentrated the values are close to a typical value versus how dispersed or spread out they are․ If most bars are narrow, depression values over the years were likely to have been fairly similar, while a tail indicates wide variation with extremes high and low․ This plot describes the distribution of the depressive variable in the dataset (e․g․, its common range, spread, and outliers) rather than a single summary statistic (like the mean) that we might have used before․ Most depression values for 18–29-year-olds cluster in the mid-to-high range, indicating a relatively high average, while the wide spread suggests substantial variability and the presence of potential outliers.


In [34]:
g = merged_data.loc[merged_data["Subgroup"].eq("18 - 29 years"), "Value"]
g = pd.to_numeric(g, errors="coerce").dropna()

g.iplot(
    kind="hist",
    bins=20,
    title="Depressive Value Distribution — 18-29 year olds (All time periods)",
    xTitle="Depressive Value",
    yTitle="Count"
)


### 2.4. Trend Comparison

*"How do depression trends over time differ between California and Florida?"*

In the CA vs FL chart, we checked the "Symptoms of Depressive Disorder" option to show the "By State" data for California and Florida, comparing the Value over time to find similarities and differences between states․ This relates to our research question since if one state had higher or increasing levels of depression during periods of more severe COVID policy implementation than another state, we may find an association between policy severity and mental health․ The first limitation of the chart is that it does not include the policy variable (StringencyIndex, for example)․ Second, there are two states and several alternative explanations (differences in demographics, number of cases/deaths, economy, etc․)․ To answer the question more directly, we need to have depression data together with the policy variable for many states and also estimate the timing and confounders of the effects․ Visually, both states follow a broadly similar downward trend after 2021 with short-term fluctuations, suggesting comparable temporal patterns despite differences in policy approaches.

In [ ]:
x = merged_data[(merged_data["Indicator"]=="Symptoms of Depressive Disorder") &
                (merged_data["Group"]=="By State") &
                (merged_data["State"].isin(["California","Florida"]))].copy()

x["date"] = pd.to_datetime(x["Time Period Start Date"].astype(str), format="%Y%m%d", errors="coerce")
x["Value"] = pd.to_numeric(x["Value"], errors="coerce")
wide = x.dropna(subset=["date","Value"]).pivot_table(index="date", columns="State", values="Value", aggfunc="mean").sort_index()

p = wide[["California","Florida"]].reset_index()
p["date"] = p["date"].dt.strftime("%Y-%m-%d")

p.iplot(kind="line", x="date", y=["California","Florida"],
        title="Symptoms of Depressive Disorder — CA vs FL",
        xTitle="Date", yTitle="Value (%)") # type: ignore